In [ ]:
import torch
import json
from pathlib import Path
from pydantic_settings import BaseSettings
import logging
from transformers import CLIPProcessor, CLIPModel


logger = logging.getLogger(__name__)
logging.basicConfig(filename='example.log', encoding='utf-8', level=logging.DEBUG)

class Settings(BaseSettings):
    # API settings
    api_title: str = "Digital Collections Explorer API"
    api_description: str = "API for searching collections using CLIP embeddings"
    api_version: str = "0.1.0"
    host: str = "0.0.0.0"
    port: int = 8000
    debug: bool = True
    
    # CLIP model settings
    clip_model: str = "openai/clip-vit-base-patch32"
    device: str = "cuda"
    batch_size: int = 32
    
    # Data directories
    collection_type: str = "photographs" # this is the default collection type, will be overwritten by config.json
    raw_data_dir: str = "data/raw"
    processed_data_dir: str = "data/processed"
    embeddings_dir: str = "data/embeddings"
    thumbnails_dir: str = "data/thumbnails"

class CLIPService:
    def __init__(self):
        self.device = "cuda"
        if self.device == "cuda" and not torch.cuda.is_available():
            logger.warning("CUDA not available, using CPU instead")
            self.device = "cpu"
        
        self.model_name = Settings.clip_model
        self.model = None
        self.processor = None
        self.load_model()
    
    def load_model(self):
        """Load CLIP model and processor"""
        logger.info(f"Loading CLIP model: {self.model_name}")
        try:
            self.model = CLIPModel.from_pretrained(self.model_name).to(self.device)
            self.processor = CLIPProcessor.from_pretrained(self.model_name)
            self.model.eval()
            logger.info(f"CLIP model loaded successfully")
        except Exception as e:
            logger.error(f"Error loading CLIP model: {str(e)}")
            raise
    
    def encode_text(self, texts) -> torch.Tensor:
        """Encode text to embedding"""
        text_inputs = self.processor(text=texts, return_tensors="pt", padding=True)
        text_inputs = {k: v.to(self.device) for k, v in text_inputs.items()}
        
        with torch.no_grad():
            text_features = self.model.get_text_features(**text_inputs)
            text_features = text_features / text_features.norm(dim=-1, keepdim=True)
        
        return text_features.cpu()
    
    def encode_image(self, image) -> torch.Tensor:
        """Encode image to embedding"""
        image_inputs = self.processor(images=image, return_tensors="pt", padding=True)
        image_inputs = {k: v.to(self.device) for k, v in image_inputs.items()}

        with torch.no_grad():
            image_features = self.model.get_image_features(**image_inputs)
            image_features = image_features / image_features.norm(dim=-1, keepdim=True)
        return image_features.cpu()

In [ ]:
    def search(self, query_embedding: torch.Tensor, logit_scale: Optional[float] = None, limit: int = 20, offset: int = 0) -> List[Dict[str, Any]]:
        """Search for similar items using query embedding with pagination"""
        try:
            similarities = torch.matmul(self.embeddings, query_embedding.t()).squeeze()

            if logit_scale is not None:
                similarities = similarities * logit_scale
            
            top_k = min(offset + limit, len(similarities))
            top_scores, top_indices = torch.topk(similarities, k=top_k)

            start_idx = min(offset, len(top_indices))
            end_idx = min(offset + limit, len(top_indices))
            
            paginated_indices = top_indices[start_idx:end_idx]
            paginated_scores = top_scores[start_idx:end_idx]
            
            results = []
            
            for idx, score in zip(paginated_indices.tolist(), paginated_scores.tolist()):
                idx_int = int(idx)
                if idx_int >= len(self.item_ids):
                    logger.warning(f"Index {idx_int} out of range for item_ids of length {len(self.item_ids)}")
                    continue
                    
                item_id = self.item_ids[idx_int]
                metadata = self.metadata.get(item_id, {})
                
                result = {
                    "id": item_id,
                    "score": float(score),
                    "metadata": metadata
                }
                
                results.append(result)
            
            return results
            
        except Exception as e:
            logger.error(f"Error in search: {str(e)}")
            logger.error(traceback.format_exc())
            return []